# Notebook 7: Live Data from Jolpica

The Kaggle snapshot stops at round 3 of 2026. Here the free Jolpica API (no key needed) fills in the rest, so the system can answer questions about this season.

The code is in `src/live_data.py`, with checks in `tests/test_live_data.py`. Running this notebook again is safe: it replaces the 2026 data each time.

In [ ]:
import sys
from pathlib import Path

import duckdb

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.live_data import (DB_PATH, apply_season, backup_database, fetch_season, latest_round,
                           normalize, rebuild_index, validate, write_corpus)

SEASON = 2026

before = duckdb.connect(str(DB_PATH), read_only=True)
print("Stored rounds:", before.execute("SELECT COUNT(DISTINCT round) FROM f1_results WHERE season = ?", [SEASON]).fetchone()[0])
print("Result rows:  ", before.execute("SELECT COUNT(*) FROM f1_results").fetchone()[0])

## Fetch

The API returns pages of 100 rows. A single race can span two pages, so rows are flattened before use. Requests are spaced out to stay inside the free rate limit.

In [ ]:
tables = normalize(fetch_season(SEASON), SEASON)
last_round = latest_round(tables)
season_finished = last_round == tables["f1_races"]["round"].max()

print({name: len(table) for name, table in tables.items()})
print(f"Latest round with results: {last_round} | season finished: {season_finished}")

## Validate before writing

The API is outside our control, so nothing is written until the data passes these checks. The strongest one: rounds we already have must match what the API says now. If two sources disagree, a human should look.

In [ ]:
problems = validate(tables, before, SEASON)
before.close()

print(problems or "No problems found")
assert not problems, "Stopping before any write"

## Merge

The whole season is replaced in one transaction. Appending would be fragile: the calendar changed from 22 to 23 rounds, so old race rows would stay wrong. A transaction means a failure leaves the database as it was.

In [ ]:
print("Backup:", backup_database())
print("New lookup rows:", apply_season(tables, SEASON))

con = duckdb.connect(str(DB_PATH), read_only=True)
print("Result rows now:", con.execute("SELECT COUNT(*) FROM f1_results").fetchone()[0])
print(con.execute("""
    SELECT round, race_name, driver_name, constructor
    FROM f1_results WHERE season = ? AND position = 1
    ORDER BY round DESC LIMIT 3
""", [SEASON]).df())
print(con.execute("""
    SELECT position, driver_id, points, wins
    FROM f1_driver_standings WHERE season = ?
    ORDER BY position LIMIT 3
""", [SEASON]).df())
con.close()

## Rebuild the corpus and index

Corpus line numbers are the Chroma ids, so replacing lines in the middle would shift every id after them. A full rebuild takes minutes and can't leave the two out of step.

Standings now say "After round N" while a season is running. The old template said "At the end of the 2026 season", which would be false.

In [ ]:
sentences = write_corpus(SEASON, last_round, season_finished)
print(f"Corpus: {len(sentences)} sentences")
print([s for s in sentences if f"After round" in s][0])
print("Indexed:", rebuild_index(sentences))

## Does the system know about it?

A quick check that new data reaches answers through both paths. This isn't an evaluation: Notebook 8 does that.

In [ ]:
from src.router import ask
from src.text_to_sql import con as read_con

truth = read_con.execute("""
    SELECT driver_name FROM f1_results
    WHERE season = 2026 AND race_name = 'United States Grand Prix' AND position = 1
""").fetchone()[0]
wins = read_con.execute("""
    SELECT COUNT(*) FROM f1_results
    WHERE season = 2026 AND driver_name = 'Max Verstappen' AND position = 1
""").fetchone()[0]
print(f"Expected: {truth} won the US Grand Prix, Verstappen has {wins} wins\n")

for question in [
    "Who won the 2026 United States Grand Prix?",
    "How many races did Max Verstappen win in 2026?",
    "Who leads the 2026 drivers championship?",
    "Tell me about the 2026 Singapore Grand Prix",
    "How did Mercedes do in the 2026 constructors championship?",
]:
    result = ask(question)
    print(f"[{result['path']}] {question}\n  {result['answer'][:200]}\n")